In [2]:
import rasterio
import numpy as np

vars = ["Aridity_Index", "ET", "GPP", "LST", "NDVI", "S2_B11", "S2_B12", "S2_B2", "S2_B3", "S2_B4", "S2_B8", "Slope", "TWI", "bio01", "bio02", "bio03", "bio04", "bio05", "bio06", "bio07", "bio08", "bio09", "bio10", "bio11", "bio12", "bio13", "bio14", "bio15", "bio16", "bio17", "bio18", "bio19"]
len(vars)



32

In [ ]:
feature_files = {
    var: f"path/to/{var}.tif" for var in vars}

# Read rasters into a dict
rasters = {}
for name, f in feature_files.items():
    with rasterio.open(f) as src:
        rasters[name] = src.read(1)  # read first band
        profile = src.profile  # save metadata for writing later

In [1]:
import joblib
xgb_model = joblib.load("./inference/model.pkl")


In [ ]:
columns = list(feature_files.keys())
rows, cols = rasters[columns[0]].shape
data = np.stack([rasters[col].ravel() for col in columns], axis=1)  # shape: (n_pixels, n_features)


In [ ]:
mask = np.any([rasters[col] == profile['nodata'] for col in columns], axis=0).ravel()
data_valid = data[~mask]


In [ ]:
y_pred = np.zeros(data.shape[0], dtype=np.float32)
y_pred[~mask] = xgb_model.predict(data_valid)
y_pred[mask] = np.nan  # keep nodata as NaN


In [ ]:
with rasterio.open("predicted_raster.tif", "w", **profile) as dst:
    dst.write(y_raster, 1)  # write as single band
